# CinemaScope — IMDb Movie Analytics Pipeline

End-to-end data pipeline using PySpark, Delta Lake, and SQL. This notebook ingests IMDb bulk datasets, builds Bronze/Silver/Gold layers, performs analytics, and validates data quality.

## 1. Setup and Raw Data

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.cinemascope_raw;

In [0]:
display(dbutils.fs.ls(
    "/Volumes/workspace/default/cinemascope_raw/"
))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/cinemascope_raw/bronze_ratings/,bronze_ratings/,0,1791804066765
dbfs:/Volumes/workspace/default/cinemascope_raw/gold_director_stats/,gold_director_stats/,0,1791804066765
dbfs:/Volumes/workspace/default/cinemascope_raw/gold_movie_rankings/,gold_movie_rankings/,0,1791804066765
dbfs:/Volumes/workspace/default/cinemascope_raw/gold_ratings/,gold_ratings/,0,1791804066765
dbfs:/Volumes/workspace/default/cinemascope_raw/silver_ratings/,silver_ratings/,0,1791804066765
dbfs:/Volumes/workspace/default/cinemascope_raw/title.basics.tsv.gz,title.basics.tsv.gz,228478099,1791802088000
dbfs:/Volumes/workspace/default/cinemascope_raw/title.crew.tsv.gz,title.crew.tsv.gz,83556892,1791802080000
dbfs:/Volumes/workspace/default/cinemascope_raw/title.ratings.tsv.gz,title.ratings.tsv.gz,8689374,1791802033000


## 2. Ratings Pipeline: Bronze and Silver

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

ratings_schema = StructType([
    StructField("tconst", StringType(), True),
    StructField("averageRating", DoubleType(), True),
    StructField("numVotes", IntegerType(), True)
])

ratings_path = "/Volumes/workspace/default/cinemascope_raw/title.ratings.tsv.gz"

ratings_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .schema(ratings_schema)
    .csv(ratings_path)
)

display(ratings_df.limit(10))

tconst,averageRating,numVotes
tt0000001,5.7,2233
tt0000002,5.4,327
tt0000003,6.4,2393
tt0000004,5.0,203
tt0000005,6.2,3098
tt0000006,5.1,226
tt0000007,5.3,964
tt0000008,5.3,2426
tt0000009,5.3,240
tt0000010,6.8,8462


In [0]:
bronze_path = "/Volumes/workspace/default/cinemascope_raw/bronze_ratings"

(
    ratings_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

print("Bronze ratings table created successfully.")

Bronze ratings table created successfully.


In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta data
bronze_ratings = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/bronze_ratings"
)

# Clean and validate
silver_ratings = (
    bronze_ratings
    .dropDuplicates(["tconst"])
    .filter(F.col("tconst").isNotNull())
    .filter(F.col("averageRating").between(0, 10))
    .filter(F.col("numVotes") >= 0)
    .withColumn("averageRating", F.col("averageRating").cast("double"))
    .withColumn("numVotes", F.col("numVotes").cast("long"))
)

# Save the Silver Delta data
silver_path = "/Volumes/workspace/default/cinemascope_raw/silver_ratings"

(
    silver_ratings.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Silver ratings layer created successfully.")

Silver ratings layer created successfully.


In [0]:
silver_ratings = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/silver_ratings"
)

silver_ratings.select(
    F.count("*").alias("total_titles"),
    F.countDistinct("tconst").alias("unique_titles"),
    F.avg("averageRating").alias("mean_rating"),
    F.max("numVotes").alias("maximum_votes")
).show(truncate=False)

+------------+-------------+------------------+-------------+
|total_titles|unique_titles|mean_rating       |maximum_votes|
+------------+-------------+------------------+-------------+
|1719798     |1719798      |6.9727115044891645|3250919      |
+------------+-------------+------------------+-------------+



In [0]:
from pyspark.sql import functions as F

silver_path = "/Volumes/workspace/default/cinemascope_raw/silver_ratings"
gold_path = "/Volumes/workspace/default/cinemascope_raw/gold_ratings"

silver = spark.read.format("delta").load(silver_path)

# Dataset-wide average rating
C = silver.agg(F.avg("averageRating")).first()[0]

# Minimum vote threshold: 90th percentile of vote counts
m = silver.approxQuantile("numVotes", [0.90], 0.01)[0]

# Calculate the Bayesian-style weighted rating
gold = (
    silver
    .withColumn(
        "weightedRating",
        (F.col("numVotes") / (F.col("numVotes") + F.lit(m)))
        * F.col("averageRating")
        + (F.lit(m) / (F.col("numVotes") + F.lit(m)))
        * F.lit(C)
    )
    .withColumn("voteThreshold", F.lit(m))
)

(
    gold.write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)

print(f"Gold layer created. Average rating: {C:.3f}")
print(f"Vote threshold (90th percentile): {m}")

Gold layer created. Average rating: 6.973
Vote threshold (90th percentile): 485.0


## 3. Gold Ratings and Weighted Scores

In [0]:
gold = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/gold_ratings"
)

display(
    gold
    .filter(F.col("numVotes") >= F.col("voteThreshold"))
    .orderBy(F.desc("weightedRating"))
    .limit(20)
)

tconst,averageRating,numVotes,weightedRating,voteThreshold
tt4283088,9.9,315537,9.895507480743992,485.0
tt4283094,9.9,233582,9.893934493455626,485.0
tt2178784,9.9,188416,9.892484238197142,485.0
tt34952922,9.9,100808,9.88598387923823,485.0
tt12187040,9.9,90006,9.884310761066603,485.0
tt39848784,9.9,53032,9.873471328356933,485.0
tt8084176,9.9,48935,9.871272057460097,485.0
tt11294360,9.9,45863,9.869367935610555,485.0
tt11028174,9.9,34729,9.85968265688868,485.0
tt1204265,9.9,29435,9.852548966566777,485.0


## 4. Movie Metadata and Gold Movie Rankings

In [0]:
from pyspark.sql import functions as F

basics_path = (
    "/Volumes/workspace/default/cinemascope_raw/title.basics.tsv.gz"
)

basics_schema = """
    tconst STRING,
    titleType STRING,
    primaryTitle STRING,
    originalTitle STRING,
    isAdult INT,
    startYear STRING,
    endYear STRING,
    runtimeMinutes STRING,
    genres STRING
"""

basics = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .schema(basics_schema)
    .csv(basics_path)
    .filter(F.col("titleType") == "movie")
    .select(
        "tconst",
        "primaryTitle",
        "startYear",
        "runtimeMinutes",
        "genres"
    )
)

print("Movie metadata DataFrame created.")

Movie metadata DataFrame created.


In [0]:
from pyspark.sql import functions as F

gold = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/gold_ratings"
)

# Convert IMDb's missing-value marker to NULL safely
basics_clean = (
    basics
    .withColumn(
        "startYear",
        F.expr("try_cast(nullif(startYear, '\\\\N') AS INT)")
    )
    .withColumn(
        "runtimeMinutes",
        F.expr("try_cast(nullif(runtimeMinutes, '\\\\N') AS INT)")
    )
)

movie_gold = gold.join(
    basics_clean,
    on="tconst",
    how="inner"
)

display(
    movie_gold
    .filter(F.col("numVotes") >= F.col("voteThreshold"))
    .orderBy(F.desc("weightedRating"))
    .select(
        "primaryTitle",
        "startYear",
        "genres",
        "averageRating",
        "numVotes",
        "weightedRating"
    )
    .limit(20)
)

primaryTitle,startYear,genres,averageRating,numVotes,weightedRating
Om Ka Hari,2026,"Comedy,Drama",9.9,5021,9.64214767157246
The Shawshank Redemption,1994,Drama,9.3,3250919,9.299652846917725
Anakapalli,2026,"Action,Crime,Drama",9.9,1616,9.224257534354097
The Godfather,1972,"Crime,Drama",9.2,2263773,9.199522918801513
Anbil Avan,2026,"Action,Drama,Romance",9.7,2107,9.189685601727607
Oye Chill Maar,2026,"Comedy,Drama",9.7,2087,9.185717371569968
Manjar,2026,Thriller,9.6,2455,9.166586761795225
Anireekshitha Atithigalu,2026,"Drama,Thriller",9.8,1646,9.156529835606738
Happy Journey,2026,"Comedy,Drama",9.7,1724,9.101206464317771
The Dark Knight,2008,"Crime,Thriller",9.1,3237851,9.099681399669361


In [0]:
movie_gold_path = (
    "/Volumes/workspace/default/cinemascope_raw/gold_movie_rankings"
)

(
    movie_gold.write
    .format("delta")
    .mode("overwrite")
    .save(movie_gold_path)
)

print("Gold movie rankings saved.")

Gold movie rankings saved.


In [0]:
from pyspark.sql import functions as F

movies = spark.read.format("delta").load(movie_gold_path)

eligible_movies = movies.filter(
    F.col("numVotes") >= F.col("voteThreshold")
)

genre_stats = (
    eligible_movies
    .filter(F.col("genres").isNotNull())
    .withColumn("genre", F.explode(F.split("genres", ",")))
    .groupBy("genre")
    .agg(
        F.count("*").alias("movie_count"),
        F.round(F.avg("averageRating"), 2).alias("avg_rating"),
        F.round(F.avg("weightedRating"), 2).alias("avg_weighted_rating"),
        F.round(F.avg("numVotes"), 0).alias("avg_votes")
    )
    .filter(F.col("genre") != r"\N")
    .orderBy(F.desc("avg_weighted_rating"))
)

display(genre_stats)

genre,movie_count,avg_rating,avg_weighted_rating,avg_votes
News,60,7.29,7.24,3539.0
Documentary,4466,7.13,7.1,3424.0
Biography,3433,6.84,6.89,24024.0
History,2653,6.75,6.82,13745.0
Film-Noir,618,6.69,6.81,6719.0
War,1739,6.63,6.76,16809.0
Music,2117,6.64,6.73,12396.0
Animation,2173,6.5,6.63,36247.0
Reality-TV,4,6.32,6.61,786.0
Musical,1333,6.43,6.6,9101.0


## 5. Genre and Decade Analytics

In [0]:
decade_stats = (
    eligible_movies
    .filter(F.col("startYear").isNotNull())
    .withColumn(
        "decade",
        (F.floor(F.col("startYear") / 10) * 10).cast("int")
    )
    .groupBy("decade")
    .agg(
        F.count("*").alias("movie_count"),
        F.round(F.avg("averageRating"), 2).alias("avg_rating"),
        F.round(F.avg("weightedRating"), 2).alias("avg_weighted_rating")
    )
    .orderBy("decade")
)

display(decade_stats)

decade,movie_count,avg_rating,avg_weighted_rating
1890,1,5.3,6.04
1900,2,6.25,6.48
1910,129,6.39,6.61
1920,417,6.81,6.91
1930,1445,6.56,6.73
1940,1753,6.64,6.77
1950,2525,6.53,6.68
1960,3103,6.46,6.63
1970,3984,6.26,6.49
1980,4734,6.16,6.37


## 6. Director Analytics

In [0]:
from pyspark.sql import functions as F

crew_path = (
    "/Volumes/workspace/default/cinemascope_raw/title.crew.tsv.gz"
)

crew_schema = """
    tconst STRING,
    directors STRING,
    writers STRING
"""

crew_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .schema(crew_schema)
    .csv(crew_path)
    .select("tconst", "directors")
)

display(crew_df.limit(10))

tconst,directors
tt0000001,nm0005690
tt0000002,nm0721526
tt0000003,nm0721526
tt0000004,nm0721526
tt0000005,nm0005690
tt0000006,nm0005690
tt0000007,"nm0005690,nm0374658"
tt0000008,nm0005690
tt0000009,nm0085156
tt0000010,nm0525910


In [0]:
movies = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/gold_movie_rankings"
)

# Keep titles with sufficient audience votes
eligible_movies = movies.filter(
    F.col("numVotes") >= F.col("voteThreshold")
)

director_stats = (
    eligible_movies
    .join(crew_df, on="tconst", how="inner")
    .filter(
        F.col("directors").isNotNull()
        & (F.col("directors") != r"\N")
    )
    .withColumn("director_id", F.explode(F.split("directors", ",")))
    .groupBy("director_id")
    .agg(
        F.countDistinct("tconst").alias("eligible_movie_count"),
        F.round(F.avg("averageRating"), 2).alias("avg_rating"),
        F.round(F.avg("weightedRating"), 2).alias("avg_weighted_rating"),
        F.sum("numVotes").alias("total_votes")
    )
    .filter(F.col("eligible_movie_count") >= 3)
    .orderBy(F.desc("avg_weighted_rating"))
)

display(director_stats.limit(20))

director_id,eligible_movie_count,avg_rating,avg_weighted_rating,total_votes
nm0606835,3,8.73,8.48,18833
nm0002160,3,8.4,8.36,85804
nm6281092,3,8.5,8.26,8409
nm0027974,3,8.53,8.25,6863
nm2336554,3,8.2,8.19,311263
nm0881279,5,8.18,8.18,4732232
nm0634240,13,8.18,8.18,19252360
nm9284362,3,8.27,8.17,20444
nm10201503,8,8.36,8.16,30186
nm0797672,4,8.23,8.15,42779


In [0]:
director_gold_path = (
    "/Volumes/workspace/default/cinemascope_raw/gold_director_stats"
)

(
    director_stats
    .write
    .format("delta")
    .mode("overwrite")
    .save(director_gold_path)
)

print("Director analytics saved successfully.")

Director analytics saved successfully.


In [0]:
saved_directors = spark.read.format("delta").load(
    "/Volumes/workspace/default/cinemascope_raw/gold_director_stats"
)

display(saved_directors.limit(20))

director_id,eligible_movie_count,avg_rating,avg_weighted_rating,total_votes
nm0606835,3,8.73,8.48,18833
nm0002160,3,8.4,8.36,85804
nm6281092,3,8.5,8.26,8409
nm0027974,3,8.53,8.25,6863
nm2336554,3,8.2,8.19,311263
nm0634240,13,8.18,8.18,19252360
nm0881279,5,8.18,8.18,4732232
nm9284362,3,8.27,8.17,20444
nm10201503,8,8.36,8.16,30186
nm0797672,4,8.23,8.15,42779


## 7. Register Gold Tables for SQL

In [0]:
gold_tables = {
    "cinemascope_movie_rankings": (
        "/Volumes/workspace/default/cinemascope_raw/gold_movie_rankings"
    ),
    "cinemascope_director_stats": (
        "/Volumes/workspace/default/cinemascope_raw/gold_director_stats"
    ),
    "cinemascope_ratings": (
        "/Volumes/workspace/default/cinemascope_raw/gold_ratings"
    )
}

for table_name, path in gold_tables.items():
    df = spark.read.format("delta").load(path)
    df.write.format("delta").mode("overwrite").saveAsTable(
        f"workspace.default.{table_name}"
    )

print("CinemaScope Gold tables created.")

CinemaScope Gold tables created.


## 8. SQL Analytics

In [0]:
%sql
SELECT
    primaryTitle,
    startYear,
    genres,
    ROUND(averageRating, 2) AS imdb_rating,
    numVotes,
    ROUND(weightedRating, 2) AS weighted_rating
FROM workspace.default.cinemascope_movie_rankings
WHERE numVotes >= voteThreshold
ORDER BY weightedRating DESC
LIMIT 20;

primaryTitle,startYear,genres,imdb_rating,numVotes,weighted_rating
Om Ka Hari,2026,"Comedy,Drama",9.9,5021,9.64
The Shawshank Redemption,1994,Drama,9.3,3250919,9.3
Anakapalli,2026,"Action,Crime,Drama",9.9,1616,9.22
The Godfather,1972,"Crime,Drama",9.2,2263773,9.2
Anbil Avan,2026,"Action,Drama,Romance",9.7,2107,9.19
Oye Chill Maar,2026,"Comedy,Drama",9.7,2087,9.19
Manjar,2026,Thriller,9.6,2455,9.17
Anireekshitha Atithigalu,2026,"Drama,Thriller",9.8,1646,9.16
Happy Journey,2026,"Comedy,Drama",9.7,1724,9.1
The Dark Knight,2008,"Crime,Thriller",9.1,3237851,9.1


In [0]:
%sql
SELECT
    genre,
    COUNT(*) AS movie_count,
    ROUND(AVG(averageRating), 2) AS avg_rating,
    ROUND(AVG(weightedRating), 2) AS avg_weighted_rating
FROM (
    SELECT
        EXPLODE(SPLIT(genres, ',')) AS genre,
        averageRating,
        weightedRating
    FROM workspace.default.cinemascope_movie_rankings
    WHERE numVotes >= voteThreshold
      AND genres IS NOT NULL
      AND genres != '\\N'
) g
GROUP BY genre
HAVING COUNT(*) >= 100
ORDER BY avg_weighted_rating DESC;

genre,movie_count,avg_rating,avg_weighted_rating
Documentary,4466,7.13,7.1
Biography,3433,6.84,6.89
History,2653,6.75,6.82
Film-Noir,618,6.69,6.81
War,1739,6.63,6.76
Music,2117,6.64,6.73
Animation,2173,6.5,6.63
Musical,1333,6.43,6.6
Sport,1288,6.44,6.58
Drama,39235,6.35,6.53


In [0]:
%sql
SELECT
    FLOOR(startYear / 10) * 10 AS decade,
    COUNT(*) AS movie_count,
    ROUND(AVG(averageRating), 2) AS avg_rating,
    ROUND(AVG(weightedRating), 2) AS avg_weighted_rating
FROM workspace.default.cinemascope_movie_rankings
WHERE numVotes >= voteThreshold
  AND startYear IS NOT NULL
GROUP BY FLOOR(startYear / 10) * 10
ORDER BY decade;

decade,movie_count,avg_rating,avg_weighted_rating
1890,1,5.3,6.04
1900,2,6.25,6.48
1910,129,6.39,6.61
1920,417,6.81,6.91
1930,1445,6.56,6.73
1940,1753,6.64,6.77
1950,2525,6.53,6.68
1960,3103,6.46,6.63
1970,3984,6.26,6.49
1980,4734,6.16,6.37


In [0]:
%sql
SELECT
    director_id,
    eligible_movie_count,
    avg_rating,
    avg_weighted_rating,
    total_votes
FROM workspace.default.cinemascope_director_stats
ORDER BY eligible_movie_count DESC, avg_weighted_rating DESC
LIMIT 20;

director_id,eligible_movie_count,avg_rating,avg_weighted_rating,total_votes
nm0002031,85,6.68,6.8,1093341
nm0001238,70,4.71,5.44,96057
nm0000406,67,6.82,6.92,770924
nm0909825,60,6.65,6.75,237085
nm0503777,55,6.75,6.86,231418
nm0002030,55,6.69,6.81,940590
nm0861703,55,6.37,6.57,611195
nm0000033,54,6.84,6.9,4308327
nm0698184,53,6.75,6.82,387490
nm0920074,53,6.66,6.78,201003


## 9. Data Quality Checks

In [0]:

from pyspark.sql import functions as F

movies = spark.table("workspace.default.cinemascope_movie_rankings")

# Aggregate basic quality metrics
metrics = movies.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("tconst").alias("unique_title_ids"),
    F.sum(
        F.when(F.col("tconst").isNull(), 1).otherwise(0)
    ).alias("null_title_ids"),
    F.sum(
        F.when(
            F.col("averageRating").isNull()
            | (F.col("averageRating") < 0)
            | (F.col("averageRating") > 10),
            1
        ).otherwise(0)
    ).alias("invalid_ratings"),
    F.sum(
        F.when(
            F.col("numVotes").isNull() | (F.col("numVotes") < 0),
            1
        ).otherwise(0)
    ).alias("invalid_vote_counts"),
    F.sum(
        F.when(F.col("weightedRating").isNull(), 1).otherwise(0)
    ).alias("null_weighted_ratings")
).first()

checks = [
    ("No null title IDs", metrics["null_title_ids"] == 0,
     metrics["null_title_ids"]),
    ("No duplicate title IDs",
     metrics["total_rows"] == metrics["unique_title_ids"],
     metrics["total_rows"] - metrics["unique_title_ids"]),
    ("Ratings between 0 and 10", metrics["invalid_ratings"] == 0,
     metrics["invalid_ratings"]),
    ("No invalid vote counts", metrics["invalid_vote_counts"] == 0,
     metrics["invalid_vote_counts"]),
    ("No null weighted ratings", metrics["null_weighted_ratings"] == 0,
     metrics["null_weighted_ratings"])
]

results = [
    (name, "PASS" if passed else "FAIL", int(value or 0))
    for name, passed, value in checks
]

quality_df = spark.createDataFrame(
    results,
    ["check_name", "status", "failed_records"]
)

display(quality_df)


check_name,status,failed_records
No null title IDs,PASS,0
No duplicate title IDs,PASS,0
Ratings between 0 and 10,PASS,0
No invalid vote counts,PASS,0
No null weighted ratings,PASS,0


In [0]:
(
    quality_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.cinemascope_quality_checks")
)

In [0]:
%sql
SELECT *
FROM workspace.default.cinemascope_quality_checks
ORDER BY status, check_name;

check_name,status,failed_records
No duplicate title IDs,PASS,0
No invalid vote counts,PASS,0
No null title IDs,PASS,0
No null weighted ratings,PASS,0
Ratings between 0 and 10,PASS,0


## 10. Final Pipeline Summary

In [0]:
%sql
SELECT
    COUNT(*) AS movie_records,
    COUNT(DISTINCT tconst) AS unique_titles,
    ROUND(AVG(averageRating), 2) AS average_rating,
    ROUND(AVG(weightedRating), 2) AS average_weighted_rating,
    MAX(numVotes) AS maximum_votes
FROM workspace.default.cinemascope_movie_rankings;

movie_records,unique_titles,average_rating,average_weighted_rating,maximum_votes
349988,349988,6.15,6.73,3250919


In [0]:
%sql
SELECT check_name, status, failed_records
FROM workspace.default.cinemascope_quality_checks
ORDER BY check_name;

check_name,status,failed_records
No duplicate title IDs,PASS,0
No invalid vote counts,PASS,0
No null title IDs,PASS,0
No null weighted ratings,PASS,0
Ratings between 0 and 10,PASS,0
